In [1]:
list.of.packages <- c("tidyverse","data.table","stargazer")
new.packages <- list.of.packages[!(list.of.packages %in% installed.packages()[,"Package"])]
if(length(new.packages)) install.packages(new.packages, repos = "http://cran.us.r-project.org")


invisible(lapply(list.of.packages, library, character.only = TRUE))

options(repr.matrix.max.rows=500, repr.matrix.max.cols=1200, message=FALSE, warning=TRUE)  

── Attaching core tidyverse packages ─────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ───────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attachement du package : ‘data.table’


Les objets suivants sont masqués depuis ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


Les objets suivants sont masqués depuis ‘package:dplyr’:

    between, first, last


L'objet suivant est masqué depuis ‘package:purrr’:

    transpose



Please 

# Longer dataframe

In [2]:
df_deg_bins <- fread("./interm/df_T_cells_CDC_longer.csv")
sprintf("%i x %i dataframe", nrow(df_deg_bins), ncol(df_deg_bins))
head(df_deg_bins,2)

[1] "54270555 x 7 dataframe"

Measured_date_CDC,cell_x,cell_y,Temp_bin,N_hours,Temp,Day_Min
<int>,<dbl>,<dbl>,<chr>,<int>,<int>,<int>
42474,91.9,27.6,T0DayMin61,3,0,61
42473,91.9,27.6,T0DayMin61,4,0,61


In [3]:
df_deg_bins%>%count(Day_Min)

Day_Min,n
<int>,<int>
61,1803481
62,1804867
63,1806547
64,1807270
65,1806454
66,1805860
67,1805450
68,1804412
69,1805029


In [4]:
df_deg_bins%>%count(N_hours)

N_hours,n
<int>,<int>
1,19048233
2,18330953
3,8387996
4,4311069
5,2004206
6,909201
7,473273
8,276921
9,171160


In [5]:
df_deg_bins%>%count(Temp)

Temp,n
<int>,<int>
-31,49
-30,39
-29,106
-28,171
-27,307
-26,539
-25,875
-24,1173
-23,1679


# Computation $$T_{i}^{30}= \sum_{b} \frac{h_{ib}^{30} \times T_b}{24 \times 30}$$

## 30-day exposure

In [6]:
result_30 <- df_deg_bins[, .(T_i61_90 = sum(N_hours * Temp / 24 / 30, na.rm = TRUE)), 
                          by = .(Measured_date_CDC, cell_x, cell_y)]
sprintf("%i x %i dataframe", nrow(result_30), ncol(result_30))
head(result_30,2)

[1] "173657 x 4 dataframe"

Measured_date_CDC,cell_x,cell_y,T_i61_90
<int>,<dbl>,<dbl>,<dbl>
42474,91.9,27.6,-3.343056
42473,91.9,27.6,-3.468056


In [7]:
data.frame(result_30) %>% 
        select(T_i61_90)%>%
        stargazer(nobs=TRUE, digits = 2,
                  summary.stat = c("n","mean", "sd","min","p25","median","p75","max"),
                  title = c("Summary statistics"),
                  type="text"
                     )


Summary statistics
Statistic    N    Mean  St. Dev.  Min   Pctl(25) Median Pctl(75)  Max 
----------------------------------------------------------------------
T_i61_90  173,657 22.11   6.68   -15.96  17.93   22.82   26.86   37.32
----------------------------------------------------------------------


# Categorize temperature into bins

### Air temperature thresholds

In [8]:
thresholds_T <- c(-18,17,21,23,25,27,29,31,33,40)

In [9]:
T_bins_labels <- c("<=17°C","(17,21]","(21,23]","(23,25]","(25,27]","(27,29]","(29,31]","(31,33]",">33°C")

In [10]:
result_bins <- result_30 %>%
                    mutate(T_i61_90_bin = cut(T_i61_90, breaks = thresholds_T, 
                                           labels = T_bins_labels ,
                                           include.lowest = TRUE))
sprintf("%i x %i dataframe", nrow(result_bins), ncol(result_bins))
head(result_bins,2)

[1] "173657 x 5 dataframe"

Measured_date_CDC,cell_x,cell_y,T_i61_90,T_i61_90_bin
<int>,<dbl>,<dbl>,<dbl>,<fct>
42474,91.9,27.6,-3.343056,<=17°C
42473,91.9,27.6,-3.468056,<=17°C


## SAVE

In [11]:
fwrite(result_bins,"./output/df_T_cells_CDC_61_90.csv")